# NBA Trade Predictor Model v2.2
### STATUS: INTEGRATED MODEL LOADER & ACTIVE ANALYTICS
This notebook implements a Trade Predictor that uses predicted player outputs and Monte Carlo season simulations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import json
import pickle
import unicodedata
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
def load_trained_model(model_path='../models/final_model.pkl'):
    """
    Load the trained XGBoost model from Module 1.
    
    Returns:
        model: Trained MultiOutputRegressor
        feature_names: List of feature names
        target_names: List of target variable names
    """
    print("Loading Module 1 trained model...")
    
    if not os.path.exists(model_path):
        # Fallback to local path if running from different root
        model_path = 'models/final_model.pkl'
        if not os.path.exists(model_path):
            raise FileNotFoundError(f"Model file not found at {model_path}")
        
    with open(model_path, 'rb') as f:
        model_data = pickle.load(f)
    
    model = model_data['model']
    feature_names = model_data['feature_names']
    target_names = model_data.get('target_names', 
                                   ['target_next_ppg', 'target_next_rpg', 
                                    'target_next_apg', 'target_next_mpg', 
                                    'target_next_ts_pct'])
    
    print(f"✓ Model loaded successfully")
    print(f"  Features: {len(feature_names)}")
    print(f"  Targets: {target_names}")
    
    return model, feature_names, target_names

# Load the model assets
player_model, feature_cols, target_cols = load_trained_model()

In [ ]:
# 1. Load Data
print("Loading statistical data...")
player_df = pd.read_csv('../data/processed/player_features_v2_temporal.csv')
team_df = pd.read_csv('../data/processed/team_features_temporal.csv')

print(f"Player data shape: {player_df.shape}")
print(f"Team data shape: {team_df.shape}")

In [ ]:
# 2. Latest Player Data Preparation
latest_player_data = player_df.sort_values(['player_name', 'season']).groupby('player_name').last().reset_index()

print(f"Latest player data prepared. Unique players: {len(latest_player_data)}")

In [ ]:
# 3. Generate Player Statistical Projections
print("Generating base player projections...")
X_inference = latest_player_data[feature_cols].fillna(0)
projections = player_model.predict(X_inference)

proj_df = pd.DataFrame(projections, columns=target_cols)
proj_df['player_name'] = latest_player_data['player_name']
proj_df['team'] = latest_player_data['team']
proj_df['position'] = latest_player_data['position']

# Default injury features (to be integrated later)
for col in ['minor_count', 'moderate_count', 'severe_count', 'prev_days_missed', 'prev_severe']:
    proj_df[col] = 0

std_devs = {
    'target_next_ppg': 3.0,
    'target_next_rpg': 1.2,
    'target_next_apg': 1.0,
    'target_next_mpg': 3.5,
    'target_next_ts_pct': 0.03
}

# Normalize player names for case-insensitive matching
def normalize_name(name):
    if not isinstance(name, str): return ""
    name = unicodedata.normalize('NFKD', name).encode('ascii', 'ignore').decode('ascii')
    return name.lower().strip()

proj_df['normalized_name'] = proj_df['player_name'].apply(normalize_name)
print("Projections ready and names normalized.")

In [ ]:
# 4. Fit Heuristics and Win Prediction

def calculate_fit_penalty(roster):
    penalty = 0
    pos_counts = roster['position'].value_counts()
    if pos_counts.get('PG', 0) > 3: penalty += 0.05
    if pos_counts.get('C', 0) > 3: penalty += 0.03

    # Recalibrated threshold: 75 PPG for top 3 scorers
    top_scorers = roster.nlargest(3, 'target_next_ppg')['target_next_ppg'].sum()
    if top_scorers > 75:
        penalty += 0.08

    return penalty

def calculate_team_score(roster, sampled_stats):
    top_8_stats = sampled_stats.nlargest(8, 'target_next_ppg')
    top_8_ppg = top_8_stats['target_next_ppg'].sum()
    efficiency_bonus = top_8_stats['target_next_ts_pct'].mean() * 10

    fit_penalty = calculate_fit_penalty(roster)

    score = (top_8_ppg + efficiency_bonus) * (1 - fit_penalty)
    return score

def predict_wins(score):
    # Recalibrated formula: Mean score ~113.7 -> 41 wins
    wins = (score - 113.7) * 1.2 + 41
    return np.clip(wins, 5, 75)

def wins_to_playoff_prob(wins):
    if wins >= 42:
        return np.clip((wins - 42) * 0.05 + 0.5, 0.5, 1.0)
    else:
        return np.clip(0.5 - (42 - wins) * 0.05, 0.0, 0.5)

In [ ]:
# 5. Monte Carlo Engine

def run_simulation(roster, n_sims=1000):
    injury_probs = np.full(len(roster), 0.05) # Placeholder injury risk

    results = []
    for _ in range(n_sims):
        sampled_stats = roster.copy()
        for col, std in std_devs.items():
            sampled_stats[col] = np.maximum(0, np.random.normal(roster[col], std))

        is_injured = np.random.random(len(roster)) < injury_probs
        sampled_stats.loc[is_injured, 'target_next_ppg'] *= 0.6

        score = calculate_team_score(roster, sampled_stats)
        wins = predict_wins(score)
        results.append(wins)

    return np.array(results)

In [ ]:
def analyze_trade(team_a_id, team_b_id, outgoing_a, outgoing_b):
    """
    Executes trade analysis between two teams with robust name matching.
    """
    # Normalize inputs
    outgoing_a = [normalize_name(p) for p in outgoing_a]
    outgoing_b = [normalize_name(p) for p in outgoing_b]
    
    # Get base rosters
    roster_a = proj_df[proj_df['team'] == team_a_id].copy()
    roster_b = proj_df[proj_df['team'] == team_b_id].copy()
    
    if roster_a.empty: print(f"Warning: No players found for Team {team_a_id}")
    if roster_b.empty: print(f"Warning: No players found for Team {team_b_id}")

    # Execute trade using normalized names
    players_a = roster_a[roster_a['normalized_name'].isin(outgoing_a)]
    players_b = roster_b[roster_b['normalized_name'].isin(outgoing_b)]
    
    # Diagnostic messaging
    for p_name in outgoing_a:
        if p_name not in roster_a['normalized_name'].values:
            print(f"Warning: Player '{p_name}' not found on {team_a_id} roster!")
    for p_name in outgoing_b:
        if p_name not in roster_b['normalized_name'].values:
            print(f"Warning: Player '{p_name}' not found on {team_b_id} roster!")

    post_roster_a = pd.concat([roster_a[~roster_a['normalized_name'].isin(outgoing_a)], players_b])
    post_roster_b = pd.concat([roster_b[~roster_b['normalized_name'].isin(outgoing_b)], players_a])

    print(f"Analyzing trade: {len(players_a)} players from {team_a_id} <-> {len(players_b)} players from {team_b_id}")

    results = {}
    for label, pre, post in [('Team A', roster_a, post_roster_a), ('Team B', roster_b, post_roster_b)]:
        print(f"Simulating {label}...")
        pre_results = run_simulation(pre)
        post_results = run_simulation(post)
        results[label] = (pre_results, post_results)

    # Calculate Trade Score (0-100)
    delta_a = results['Team A'][1].mean() - results['Team A'][0].mean()
    delta_b = results['Team B'][1].mean() - results['Team B'][0].mean()
    
    # Base score 50 (neutral), +/- 10 points per combined win change
    trade_score = 50 + (delta_a + delta_b) * 10
    trade_score = max(1, min(100, trade_score))
    
    # Rating assignment
    if trade_score <= 40:
        rating = "poor trade"
    elif trade_score <= 60:
        rating = "neutral"
    elif trade_score <= 80:
        rating = "good"
    else:
        rating = "amazing"
        
    results['trade_score'] = trade_score
    results['trade_rating'] = rating
    
    print(f"\nTrade Evaluation Summary:")
    print(f"  Trade Score: {trade_score:.1f}/100")
    print(f"  Rating: {rating.upper()}")

    return results


In [ ]:
# 7. Visualization and Summary

def plot_trade_impact(results, team_a_name, team_b_name):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    for i, (label, team_name) in enumerate([('Team A', team_a_name), ('Team B', team_b_name)]):
        pre, post = results[label]
        sns.histplot(pre, color='gray', alpha=0.5, label='Pre-Trade', ax=axes[i], kde=True)
        sns.histplot(post, color='blue', alpha=0.5, label='Post-Trade', ax=axes[i], kde=True)

        axes[i].set_title(f"Win Distribution Impact: {team_name}")
        axes[i].set_xlabel("Expected Wins")
        axes[i].legend()

        delta = post.mean() - pre.mean()
        axes[i].axvline(pre.mean(), color='red', linestyle='--')
        axes[i].axvline(post.mean(), color='green', linestyle='-')

        print(f"{team_name} Expected Win Change: {delta:+.2f} wins")
        print(f"{team_name} Playoff Prob Change: {wins_to_playoff_prob(post.mean()) - wins_to_playoff_prob(pre.mean()):+.1%}")

    plt.show()

In [ ]:
# 8. Example Analysis
results = analyze_trade('LAL', 'BOS', ['Lebron James'], ['Jayson Tatum'])
plot_trade_impact(results, 'LAL', 'BOS')

In [ ]:
# Utility: Search for Player Names
def search_players(query):
    query = normalize_name(query)
    matches = proj_df[proj_df['normalized_name'].str.contains(query, na=False)]
    if matches.empty:
        print("No matches found.")
    else:
        print(matches[['player_name', 'team', 'position', 'target_next_ppg']])